### Step 1: Blackbody 0D EBM

26/06/2026
We start with the most basic zero dimensional energy balance model. We consider our planet a perfect black body at this stage.

Obsv:
T_eq = 278.33 K
T_eq is exactly as expected

-----------------------------------------------------------------------------------
27/06/2026
We add an albedo to the planet. Since we are not setting up an albedo feedback yet, changing the parameter would be enough.

Obsv:
Temperature drops to 254.58K just as expected.

In [272]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar

# Constants
sigma = 5.670374419e-8  # Stefan-Boltzmann constant (W m^-2 K-4)
PI = np.pi  # Pi

# Stellar Parameters

L_star = 3.828e26 # Solar Luminosity (W)

#Planetary Parameters

Orbital_radius = 1.496e11 # Orbital radius of Earth (m)
alpha = 0.3 # Albedo of the planet
epsilon0 = 0.77 # Emissivity of the planet

C = 4.2e8 # Heat capacity of the planet (J m^-2 K^-1)

The stellar constant ($S_0$) will not be taken as a constant but will be a function of Stellar Luminance ($L_*$) and distance from the star (r)

In [273]:
def stellar_flux (luminosity, distance):
    """Returns stellar flux"""
    return luminosity / (4 * PI * distance**2)

The 0D balance condition shall be encoded now
$$
C \frac{dT}{dt}= \frac{1}{4} (1-\alpha)S_0 - \epsilon \sigma T^4
$$

At radiative equilibrium; $\frac{dT}{dt} = 0$ and thus we get
$$
\epsilon \sigma T^4 = \frac{1}{4} (1-\alpha)S_0
$$

In [274]:
def equilibrium_temperature (stellar_flux, albedo, emissivity):
    """Returns the equilibrium temperature of a planet"""
    return (stellar_flux * (1 - albedo) / (4 * sigma * emissivity))**(1/4)

In [275]:
S0 = stellar_flux(L_star, Orbital_radius)
T_eq = equilibrium_temperature(S0, alpha, epsilon0)

print(f"S0 = {S0:.2f} W/m²")
print(f"T_eq = {T_eq:.2f} K")

S0 = 1361.13 W/m²
T_eq = 271.77 K


29/06/2026
Adding Greenhouse Effect

Obsv:
343.99K at 0.3 emissivity, that's too hot.

Instead of just tuning the parameter, we shall introduce a layer of atmosphere. The shell model.

Obsv:
$T_{s}$ comes out to be 287.48K for 30% Albedo and 77% emissivity, which is accurate for Earth.

-------
Since previous functions cannot incorporate a shell, we introduce a new function following,
$$
\frac{(1-\alpha)S_0}{4} + \epsilon \sigma T_{a}^4 = \sigma T_{s}^4
$$

Here, $T_a$ and $T_s$ are the temperatures of the atmosphere and the surface respectively. This is the **Surface Energy Balance** Equation.

Due to Greenhouse gases affecting the emissivity, we must also do an energy balance on the layer of atmosphere.
$$
\epsilon \sigma T_{s}^4 = 2\epsilon \sigma T_{a}^4
$$

This is the **Atmospheric Energy Balance** Equation.

We are only interested in the $T_{s}^4$, surface temperature, which after solving the two equations is given by the following
$$
T_{s}^4 = \left(\frac{(1-\alpha)S_{0}}{4 \sigma (1-\frac{\epsilon}{2})} \right)^\frac{1}{4}
$$

In [276]:
def surface_temp_w_greenhouse (S0, alpha, epsilon0):
    return (
        ((1-alpha)*S0)
        / (4 * sigma * (1-epsilon0/2))
    )**0.25

print(f"T_s = {surface_temp_w_greenhouse(S0, alpha, epsilon0):.2f} K")

T_s = 287.48 K


### Step 2: Adding an Atmosphere

09/07/2026
We make emissivity a function of composition and pressure to make a more realistic atmosphere.

Unfortunately, to model all the effects of the atmospheric composition on the climate, the chemistry gets sophisticated, therefore our model is limited to the effects of composition on the Greenhouse Effect only.

We use optical depth $(\tau)$ to model the emissivity $(\epsilon)$. Initially $\tau$ is treated as a constant, and then we replace it with a physically motivated expression based on composition and pressure of the atmospheric gases.

In [277]:
tau_old = 0.8  # Optical depth of the atmosphere
epsilon_old = 1 - np.exp(-tau_old)  # Emissivity of the planet

print(f"Emissivity (epsilon) = {epsilon_old:.2f}")
print(f"T_s with optical depth = {surface_temp_w_greenhouse(S0, alpha, epsilon_old):.2f} K")

Emissivity (epsilon) = 0.55
T_s with optical depth = 275.93 K


#### Step 2.1: Optical Depth Function

04/08/2026
Now we use the compostion and partial pressure of the gases and the gravity of the planet, to make a physical equation for optical depth.

The physics invloved is as follows:

For each gas,
$$
P_i = x_i P
$$

Where $P_i$ is the partial pressure, $x_i$ is the mixing ratio, and $P$ is the atmospheric pressure (of the air parcel/column);  
then,
$$
m_i = \frac{P_i}{g}
$$
where $m_i$ is the atmospheric mass of a gas above one square meter and $g$ is the acceleration due to gravity.

then, the optical density
$$
\tau_{i} = \kappa_{i} m_i
$$

$$
\tau = \sum_{i} \tau_{i}
$$

In [278]:
composition1 = {
    "N2": 0.7552,
    "O2": 0.2314,
    "Ar": 0.0182,
    "CO2": 0.00061,
    "H2O": 0.0016,
    "CH4": 1.05e-6,
}

gravity = 9.81  # m/s²

pressure = 101325  # Pa

kappa1 = {
    "N2": 0.0,
    "O2": 0.0,
    "Ar": 0.0,
    "CO2": 0.056,
    "H2O": 0.065,
    "CH4": 1.38
} # values are unconfirmed and for demonstration purposes only

In [279]:
def optical_depth(pressure, gravity, composition, kappa):
    """Returns the optical depth of the atmosphere"""
    tau = 0.0
    tau_species = {}

    for gas, x in composition.items():

        P_i = x * pressure          # Partial pressure
        m_i = P_i / gravity         # Column mass
        tau_i = kappa[gas] * m_i    # Optical depth

        tau_species[gas] = tau_i
        tau += tau_i

    return tau, tau_species

tau, tau_species = optical_depth(pressure, gravity, composition1, kappa1)

epsilon = 1 - np.exp(-tau)

#### Some Clarification

11/08/2026

1. Even though we just calculated the optical depth we shall not use the function to calculate the temperature by making emissivity a function of optical depth.   Instead we will do a radiative balance and find the temperature at the effective radiating layer, $T_{rad}$, then we will use $\tau$ to find the Pressure at that layer, $P_{rad}$.  
Knowing the pressure and temperature at this layer, we can extrapolate along an adiabat to find the temperature at surface pressure.  
2. The dictionaries `kappa` and `composition` are still incomplete we need to add greenhouse gases with significant contribution like Water Vapour and Methane, but that is not so easy as they have varying compositions.

In [280]:
def effective_radiating_temp(S0, alpha):
    """Returns the effective radiating temperature of a planet"""
    t = (S0 * (1 - alpha) / (4 * sigma))**(1/4)
    return t

In [281]:
print("===== Atmosphere =====")
print(f"Surface Pressure : {pressure:.0f} Pa")
print(f"Gravity          : {gravity:.2f} m/s²")
print()

print("Composition:")
for gas, fraction in composition1.items():
    print(f"  {gas:<4}: {fraction:.6f}")

print()

print("Optical Depth Contributions:")
for gas, value in tau_species.items():
    print(f"  {gas:<4}: {value:.4f}")

print()
print(f"Optical depth (tau) = {tau:.8f}")
print(f"Emissivity (epsilon) = {epsilon:.8f}")
print(f"T_rad = {effective_radiating_temp(S0, alpha):.2f} K")

===== Atmosphere =====
Surface Pressure : 101325 Pa
Gravity          : 9.81 m/s²

Composition:
  N2  : 0.755200
  O2  : 0.231400
  Ar  : 0.018200
  CO2 : 0.000610
  H2O : 0.001600
  CH4 : 0.000001

Optical Depth Contributions:
  N2  : 0.0000
  O2  : 0.0000
  Ar  : 0.0000
  CO2 : 0.3528
  H2O : 1.0742
  CH4 : 0.0150

Optical depth (tau) = 1.44198593
Emissivity (epsilon) = 0.76354230
T_rad = 254.58 K


#### Step 2.2

26/08/2026

Now that we have $T_{rad}$ and $\tau$ we shall calculate $P_{rad}$ using the optical depth.

In [282]:
def radiating_pressure(surface_pressure, tau_surface):
    """Returns the pressure at the effective radiating level."""

    tau_rad = 1.0

    P_rad = surface_pressure * tau_rad / tau_surface

    if tau_surface <= tau_rad:
        return surface_pressure
        
    
    return P_rad

P_rad = radiating_pressure(pressure, tau)

print(f"Radiating pressure = {P_rad:.2f} Pa")

Radiating pressure = 70267.68 Pa


#### 2.3 Finding Temperature at the Surface  

27/08/2026

Now that we have $T_{rad}$, $P_{rad}$, and $P_{s}$, we can follow Pierrehumbert's treatment from **Principles of Planetary Climate** and extrapolate the temperature to the surface along the appropriate adiabat for a dry, adiabatic atmosphere. Since we are not modeling condensation, the assumption of a dry adiabat should work for this case.

The following function calculates the surface temperature using the method discussed.

In [283]:
def surface_temperature_dry(T_rad, P_rad, surface_pressure, R, cp):

    Ts = T_rad * (surface_pressure / P_rad)**(R / cp)

    return Ts

In [284]:
R = 287.05  # J/(kg·K) for dry air
cp = 1004.0  # J/(kg·K) for dry air

T_rad = effective_radiating_temp(S0, alpha)

T_s = surface_temperature_dry(
    T_rad,
    P_rad,
    pressure,
    R,
    cp
)

print(f"Surface temperature = {T_s:.2f} K")

Surface temperature = 282.67 K


#### More Clarifications

27/08/2026

1. The current model is running on 2 assumption. First, the gray gas assumption, that assumes that the constituent gases of the atmosphere have the same absorption coefficient ($\kappa$) for all wavelength, this shall be referred to as the Gray Gas Mean Coefficient. We use this to get the coefficients for Carbon Dioxide and Water Vapour. Secondly, the model assumes a fixed water vapor composition in the atmosphere.

2. These assumptions shall be addressed in the later iterations of the model, or in the Latitudinal EBM.

### Step 3: Water Vapor Feedback

Now we shall replace the fixed mixing ratio of the water vapor in the dictionary to 0, and then we shall use an iterative loop to implement the water vapor feedback loop.  
The Water Vapour Feedback loop is governed by 4 major interconnected physical mechanics, thermodynamic coupling, hydrostatic column mass, radiative transfer, and atmosphere convective dynamics. We have already tackled the radiative transfer effects (optical depth) and we shall not be concerned with convection right now.  

##### Thermodynamic Coupling: 

The maximum amount of water vapour air can hold depends exponentially on temperature, the relation is described by the Clausius-Clapeyron equation:  
$$
\frac{de_{sat}}{dT} = \frac{L_{v} e_{sat}}{R_{v}T^2}
$$

Integrating this yields us the August-Roche-Magnus Approximation:  
$$
e_{sat}(T_s) = 611.2 exp \Big(\frac{17.67(T_s - 273.15)}{T_s - 29.65})
$$

Assuming relative humidity, ($RH = (\frac{e_v}{e_{sat}})$) remains fairly constant, warming the surface $(T_s)$ increases the vapour pressure $(e_v)$, which in turn increases the surface mass mixing ratio $q_0 \approx 0.622 \frac{e_v}{P_s}$.

#### Effects of Condensation on Column Mass

Water Vapour condenses as air rises and cools. Its concentration decays rapidly with altitude according to the power law $q(P) = q_0 \Big(\frac{P}{P_s})^\eta$, where $\eta \approx 3$ is the water vapor's vertical decay exponent. Integrating this equation under hydrostatic equilibrium gives the total column water vapour mass:  
$$
m_{H_{2}O} = \frac{q_0 P_s}{(1+ \eta)g}
$$  
Therefore, the column-effective mixing ratio $x_{H_{2}O} = \frac{q_0}{1+\eta}$

08/09/2026

The Water Vapour Feedback has been delayed since it requires extrapolation along a moist adiabat, which is much more difficult model than a dry adiabat.

07/10/2026

#### Revisiting Water Vapor Feedback

After some study, I am ready to try and simulate water vapor feedback again.

In [285]:

def calculate_h2o_mixing_ratio(T_s, surface_pressure, RH=0.70, eta=3.0):
    """
    Calculates the column-effective water vapor mixing ratio x_H2O
    based on surface temperature T_s and Clausius-Clapeyron scaling.
    """
    # Saturation vapor pressure in Pa (August-Roche-Magnus approximation)
    e_sat = 611.2 * np.exp((17.67 * (T_s - 273.15)) / (T_s - 29.65))
    
    # Vapor pressure at given relative humidity
    e_v = min(RH * e_sat, surface_pressure)  # Cap to surface pressure
    
    # Surface mass mixing ratio q0
    q0 = min(0.622 * (e_v / surface_pressure), 1.0)  # Cap to 1
    
    # Column-effective mixing ratio x_H2O
    x_H2O = q0 / (1 + eta)
    
    return x_H2O

In [286]:
# Constants & Radiating Temperature
R = 287.05
cp = 1004.0
RH = 0.70  # relative humidity
tolerance = 1e-4  # Convergence threshold in Kelvin
max_iter = 100

T_rad = effective_radiating_temp(S0, alpha)

# Initial guess for surface temperature
T_s = surface_temperature_dry(T_rad, P_rad, pressure, R, cp)

for i in range(max_iter):
    # Update H2O mixing ratio based on current T_s
    composition1["H2O"] = calculate_h2o_mixing_ratio(T_s, pressure, RH=RH)
    
    # Recalculate optical depth with updated composition
    tau, tau_species = optical_depth(pressure, gravity, composition1, kappa1)
    
    # Recalculate radiating pressure
    P_rad = radiating_pressure(pressure, tau)
    
    # Extrapolate to find new surface temperature
    T_s_new = surface_temperature_dry(T_rad, P_rad, pressure, R, cp)
    
    # Check for convergence
    if abs(T_s_new - T_s) < tolerance:
        print(f"Converged in {i+1} iterations.")
        T_s = T_s_new
        break
        
    T_s = T_s_new

# Recalculate final emissivity for output
epsilon = 1 - np.exp(-tau)

print(f"Final Surface Temperature = {T_s:.2f} K")
print(f"Final Emissivity = {epsilon:.2f}")
print(f"Final optical depth (tau) = {tau:.8f}")

Converged in 3 iterations.
Final Surface Temperature = 254.58 K
Final Emissivity = 0.38
Final optical depth (tau) = 0.47032644


#### Observations

1. Our surface temperature is equal to a that of an atmosphere-less planet
2. This is occuring because our code is under-calculating the temperature. I assume it is occuring due to our use of dry adiabatic exponent instead of a moist adiabatic exponent for the surface temperature calculation.

#### Dynamic Moist Adiabatic Exponent $\kappa_{eff}(T_s)$

Our Surface Temperature function uses the adiabatic exponent for a dry adiabat $\kappa = \frac{R}{cp} \approx 0.286$ but since we are working with water vapor now we have condensation, and hence the release of latent heat occuring in our system, this will affect the exponent and reduces it to $\kappa_{eff} \approx 0.10 - 0.15$

We can either use a static $\kappa_{eff}$ or we can use the formula for $\kappa_{eff}$ to calculate it dynamically suring the iterative loop, we shall go with a dynamic exponent because it is more fun.

To calculate the adiabatic exponent we shall use the formula

$$
\kappa_{eff}(T_s) = \frac{R}{c_p} \left[ \frac{1 + \frac{L_v q_s}{R T_s}}{1 + \frac{\epsilon L_v^2 q_s}{c_p R T_s^2}} \right]
$$

In [291]:
def surface_temperature_moist(T_rad, P_rad, surface_pressure, kappa_eff=0.14):
    
    Ts = T_rad * (surface_pressure / P_rad) ** kappa_eff
    return Ts

In [293]:
def get_kappa_eff(T_s, q_s):
    
    R = 287.05
    cp = 1004.0
    Lv = 2.5e6  # Latent heat of vaporization J/kg
    epsilon = 0.622

    num = 1.0 + (Lv * q_s) / (R * T_s)
    den = 1.0 + (epsilon * (Lv**2) * q_s) / (cp * R * (T_s**2))

    return (R / cp) * (num / den)

In [294]:
alpha_damp = 0.15  # Damping parameter (15% new value, 85% previous value)
T_s = 288.0  # Warm initial guess

for i in range(max_iter):
    # 1. Update H2O composition (with Fix 1 caps applied)
    composition1["H2O"] = calculate_h2o_mixing_ratio(T_s, pressure, RH=0.70)

    # 2. Compute optical depth
    tau, tau_species = optical_depth(pressure, gravity, composition1, kappa1)

    # 3. Compute radiating pressure (capped at surface pressure)
    P_rad = radiating_pressure(pressure, tau)

    # 4. Calculate raw target surface temperature using moist adiabat (Fix 2)
    T_s_calc = surface_temperature_moist(T_rad, P_rad, pressure, kappa_eff= get_kappa_eff(T_s, composition1["H2O"]))

    # 5. Apply damping relaxation (Fix 3)
    T_s_new = (1 - alpha_damp) * T_s + alpha_damp * T_s_calc

    # 6. Convergence check
    if abs(T_s_new - T_s) < 1e-4:
        print(f"Converged in {i+1} iterations")
        T_s = T_s_new
        break

    T_s = T_s_new

print(f"Final Surface Temperature = {T_s:.2f} K")

Converged in 72 iterations
Final Surface Temperature = 254.58 K


#### Observations

1. The problem persists! We are still getting too low of a surface temperature.
2. I think the culprit must be our mathematical tools, instead of using the radiating level formulation, we shall explore Eddington two-stream approximation for a radiative atmosphere in thermal equilibrium.